# Naukri Scraper — Simple Version
Only uses what we learned. No advanced stuff.

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

In [7]:
# Concept 1 — Setup (exactly as taught)
def get_driver():
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()))

In [8]:
driver = get_driver()
driver.implicitly_wait(10)   # Concept 6 — implicit wait

driver.get("https://www.naukri.com/data-analyst-jobs-in-all-india")
time.sleep(4)
print("Title:", driver.title)

Title: Data Analyst Jobs In All India - 26637 Data Analyst Job Vacancies In All India - Naukri.com


In [9]:
# Print ONE card HTML so you can see the actual structure
# This helps us find correct selectors for salary / rating / reviews
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
cards = soup.select("div.srp-jobtuple-wrapper")
print("Cards found:", len(cards))

# Print first card HTML to inspect structure
print("\n--- FIRST CARD HTML ---")
print(cards[0].prettify()[:3000])   # .prettify() — Day 2 concept

Cards found: 20

--- FIRST CARD HTML ---
<div class="srp-jobtuple-wrapper" data-job-id="140526016202">
 <div class="cust-job-tuple layout-wrapper lay-2 sjw__tuple">
  <div class="row1">
   <h2>
    <a class="title" href="https://www.naukri.com/job-listings-data-analyst-immediate-joiners-preferred-capita-pune-mumbai-all-areas-1-to-5-years-140526016202" rel="noopener noreferrer" target="_blank" title="Data Analyst - (Immediate Joiners preferred)">
     Data Analyst - (Immediate Joiners preferred)
    </a>
   </h2>
   <span class="imagewrap">
    <img class="logoImage" loading="lazy" src="https://img.naukimg.com/logo_images/groups/v1/599384.gif"/>
   </span>
  </div>
  <div class="row2">
   <span class="comp-dtls-wrap">
    <a class="comp-name mw-25" href="https://www.naukri.com/capita-jobs-careers-143636" target="_blank" title="Capita">
     Capita
    </a>
    <a class="rating" href="https://www.ambitionbox.com/reviews/capita-reviews?utm_campaign=srp_ratings&amp;utm_medium=desktop&amp;u

In [10]:
MAX_PAGES = 220  # Change to 220 for full scrape

all_data = []   # one list outside loop — Hacker News pattern

for page in range(1, MAX_PAGES + 1):
    
    # URL pagination — same as books.toscrape
    url = f"https://www.naukri.com/data-analyst-jobs-in-all-india-{page}"
    driver.get(url)
    time.sleep(5)   # time.sleep — delays between requests
    
    # Selenium + BeautifulSoup combo (what we learned)
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    
    cards = soup.select("div.srp-jobtuple-wrapper")   # soup.select — Day 2
    
    if not cards:
        print(f"Page {page}: no cards — stopping")
        break
    
    for card in cards:
        
        # --- Title ---
        title_tag = card.select_one("a.title")
        title     = title_tag.text.strip() if title_tag else "N/A"
        link      = title_tag["href"]      if title_tag else "N/A"
        
        # --- Company ---
        company_tag = card.select_one("a.comp-name")
        company     = company_tag.text.strip() if company_tag else "N/A"
        
        # --- Experience ---
        exp_tag    = card.select_one(".expwdth")
        experience = exp_tag.text.strip() if exp_tag else "N/A"
        
        # --- Location ---
        loc_tag  = card.select_one(".locWdth")
        location = loc_tag.text.strip() if loc_tag else "N/A"
        
        # --- Salary ---
        # Try multiple selectors because Naukri uses different ones
        sal_tag = card.select_one(".sal-wrap span[title]")   # has title attribute
        if not sal_tag:
            sal_tag = card.select_one(".sal-wrap span")       # any span inside sal-wrap
        if not sal_tag:
            sal_tag = card.select_one(".sal-wrap")            # the wrap itself
        if sal_tag:
            salary = sal_tag.get("title") or sal_tag.text.strip()
        else:
            salary = "Not Disclosed"
        
        # --- Rating ---
        # Try class names that Naukri uses for ratings
        rat_tag = card.select_one(".ambRating")
        if not rat_tag:
            rat_tag = card.select_one(".rating")
        if not rat_tag:
            # find any span whose class contains "rating"
            rat_tag = card.find("span", class_=lambda c: c and "ating" in c)
        rating = rat_tag.text.strip() if rat_tag else "N/A"
        
        # --- Reviews ---
        rev_tag = card.select_one(".review")
        if not rev_tag:
            rev_tag = card.find("span", class_=lambda c: c and "eview" in c)
        reviews = rev_tag.text.strip() if rev_tag else "N/A"
        
        # --- Posted date ---
        date_tag = card.select_one(".job-post-day")
        posted   = date_tag.text.strip() if date_tag else "N/A"
        
        # list of dicts — Day 1 pattern
        all_data.append({
            "Title"      : title,
            "Company"    : company,
            "Experience" : experience,
            "Salary"     : salary,
            "Location"   : location,
            "Rating"     : rating,
            "Reviews"    : reviews,
            "Posted"     : posted,
            "Link"       : link
        })
    
    print(f"Page {page} — {len(cards)} jobs | Total: {len(all_data)}")

driver.quit()   # always close
print(f"Done. Total jobs: {len(all_data)}")

Page 1 — 20 jobs | Total: 20
Page 2 — 20 jobs | Total: 40
Page 3 — 20 jobs | Total: 60
Page 4 — 20 jobs | Total: 80
Page 5 — 20 jobs | Total: 100
Page 6 — 20 jobs | Total: 120
Page 7 — 20 jobs | Total: 140
Page 8 — 20 jobs | Total: 160
Page 9 — 20 jobs | Total: 180
Page 10 — 20 jobs | Total: 200
Page 11 — 20 jobs | Total: 220
Page 12 — 19 jobs | Total: 239
Page 13 — 20 jobs | Total: 259
Page 14 — 20 jobs | Total: 279
Page 15 — 20 jobs | Total: 299
Page 16 — 20 jobs | Total: 319
Page 17 — 20 jobs | Total: 339
Page 18 — 20 jobs | Total: 359
Page 19 — 20 jobs | Total: 379
Page 20 — 20 jobs | Total: 399
Page 21 — 20 jobs | Total: 419
Page 22 — 20 jobs | Total: 439
Page 23 — 20 jobs | Total: 459
Page 24 — 20 jobs | Total: 479
Page 25 — 20 jobs | Total: 499
Page 26 — 20 jobs | Total: 519
Page 27 — 20 jobs | Total: 539
Page 28 — 20 jobs | Total: 559
Page 29 — 20 jobs | Total: 579
Page 30 — 20 jobs | Total: 599
Page 31 — 20 jobs | Total: 619
Page 32 — 20 jobs | Total: 639
Page 33 — 20 jobs | T

In [11]:
# list of dicts -> DataFrame — Day 1
df = pd.DataFrame(all_data)
print(df.shape)
df.head(10)

(4398, 9)


,Title,Company,Experience,Salary,Location,Rating,Reviews,Posted,Link
0,Data Analyst - (Immediate Joiners preferred),Capita,1-5 Yrs,Not Disclosed,"Hybrid - Pune, Mumbai (All Areas)",3.5,2694 Reviews,1 week ago,https://www.naukri.com/job-listings-data-analy...
1,Data Analyst,Capgemini,4-9 Yrs,Not Disclosed,"Hybrid - Chennai, Delhi / NCR, Mumbai (All Areas)",3.6,54214 Reviews,3 weeks ago,https://www.naukri.com/job-listings-data-analy...
2,Data Analyst,Snapdeal,1-6 Yrs,Not Disclosed,Gurugram,3.5,673 Reviews,3+ weeks ago,https://www.naukri.com/job-listings-data-analy...
3,Data Analyst,Kiya.ai,3-7 Yrs,5-11 Lacs PA,"Hybrid - Bengaluru, Mumbai (All Areas)",3.9,644 Reviews,3 days ago,https://www.naukri.com/job-listings-data-analy...
4,Data Analyst - Tableau/Looker/Salesforce (Remote),Indium Software,2-5 Yrs,12-16 Lacs PA,Remote,3.9,1372 Reviews,2 weeks ago,https://www.naukri.com/job-listings-data-analy...
5,Data Analyst,MNC,0-2 Yrs,2.75-5 Lacs PA,"Pimpri-Chinchwad, Pune",N/A,N/A,1 day ago,https://www.naukri.com/job-listings-data-analy...
6,Data Analyst,MNC,0-1 Yrs,2.5-5 Lacs PA,"Pimpri-Chinchwad, Pune",N/A,N/A,5 days ago,https://www.naukri.com/job-listings-data-analy...
7,Data Analyst,MNC,0-2 Yrs,2-5 Lacs PA,"Pimpri-Chinchwad, Pune",N/A,N/A,1 week ago,https://www.naukri.com/job-listings-data-analy...
8,Data Analyst,Codinglimits Nellore,0-1 Yrs,Not Disclosed,"Hyderabad, Chennai, Bengaluru",N/A,N/A,1 day ago,https://www.naukri.com/job-listings-data-analy...
9,Data Analyst,ti Steps,0-2 Yrs,3-4 Lacs PA,"Hyderabad, Chennai, Bengaluru",3.9,18 Reviews,1 day ago,https://www.naukri.com/job-listings-data-analy...


In [12]:
df

,Title,Company,Experience,Salary,Location,Rating,Reviews,Posted,Link
0,Data Analyst - (Immediate Joiners preferred),Capita,1-5 Yrs,Not Disclosed,"Hybrid - Pune, Mumbai (All Areas)",3.5,2694 Reviews,1 week ago,https://www.naukri.com/job-listings-data-analy...
1,Data Analyst,Capgemini,4-9 Yrs,Not Disclosed,"Hybrid - Chennai, Delhi / NCR, Mumbai (All Areas)",3.6,54214 Reviews,3 weeks ago,https://www.naukri.com/job-listings-data-analy...
2,Data Analyst,Snapdeal,1-6 Yrs,Not Disclosed,Gurugram,3.5,673 Reviews,3+ weeks ago,https://www.naukri.com/job-listings-data-analy...
3,Data Analyst,Kiya.ai,3-7 Yrs,5-11 Lacs PA,"Hybrid - Bengaluru, Mumbai (All Areas)",3.9,644 Reviews,3 days ago,https://www.naukri.com/job-listings-data-analy...
4,Data Analyst - Tableau/Looker/Salesforce (Remote),Indium Software,2-5 Yrs,12-16 Lacs PA,Remote,3.9,1372 Reviews,2 weeks ago,https://www.naukri.com/job-listings-data-analy...
...,...,...,...,...,...,...,...,...,...
4393,Sr. BI Analyst,Amherst Group,8-11 Yrs,Not Disclosed,"Kolkata, Mumbai, New Delhi, Hyderabad, Pune, C...",2.1,15 Reviews,3+ weeks ago,https://www.naukri.com/job-listings-sr-bi-anal...
4394,Senior AI Analyst,Impact Analytics,3-7 Yrs,Not Disclosed,Hybrid - Bengaluru,4.0,233 Reviews,3+ weeks ago,https://www.naukri.com/job-listings-senior-ai-...
4395,Scrum Master Senior Analyst,Cigna Medical Group,4-9 Yrs,Not Disclosed,Bengaluru,3.0,43 Reviews,3+ weeks ago,https://www.naukri.com/job-listings-scrum-mast...
4396,Sr. Operations Research Analyst,GForce Consulting Solutions,2-6 Yrs,Not Disclosed,"Kolkata, Mumbai, New Delhi, Hyderabad, Pune, C...",N/A,N/A,3+ weeks ago,https://www.naukri.com/job-listings-sr-operati...


In [13]:
# Check how many N/A in each column
print("N/A counts per column:")
for col in df.columns:
    na_count = (df[col] == "N/A").sum()
    print(f"  {col}: {na_count} N/A out of {len(df)}")

N/A counts per column:
  Title: 0 N/A out of 4398
  Company: 0 N/A out of 4398
  Experience: 88 N/A out of 4398
  Salary: 0 N/A out of 4398
  Location: 16 N/A out of 4398
  Rating: 780 N/A out of 4398
  Reviews: 780 N/A out of 4398
  Posted: 0 N/A out of 4398
  Link: 0 N/A out of 4398


In [14]:
# df.to_csv — last BeautifulSoup concept
df.to_csv("naukri_jobs.csv", index=False)
print(f"Saved {len(df)} rows to naukri_jobs.csv")

Saved 4398 rows to naukri_jobs.csv


## If Rating / Reviews still show N/A

Run the cell below — it prints one card's HTML.  
Find the tag that holds rating and reviews, then update the selectors above.

In [ ]:
# Run this to inspect a card that HAS a rating
driver2 = get_driver()
driver2.get("https://www.naukri.com/data-analyst-jobs-in-all-india-1")
time.sleep(4)

html2 = driver2.page_source
soup2 = BeautifulSoup(html2, "html.parser")
cards2 = soup2.select("div.srp-jobtuple-wrapper")

# Find a card that has rating text
for card in cards2:
    text = card.get_text()
    if any(char.isdigit() for char in text[:200]):  # has some number (rating)
        print(card.prettify()[:2000])
        break

driver2.quit()